<a href="https://colab.research.google.com/github/jyryu3161/2026CADD/blob/main/week7_BindingSite_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Setting environment

In [ ]:

# PDB 구조 수정 및 Docking receptor 준비
!pip -q install pdbfixer openmm meeko rdkit scipy gemmi


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 665.9/665.9 kB 16.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.4/14.4 MB 54.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.8/31.8 MB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 27.4 MB/s eta 0:00:00


In [ ]:

import requests
from io import StringIO
from collections import Counter
from pdbfixer import PDBFixer
from openmm.app import PDBFile

# ==============================
# 사용자 설정
# ==============================
PDB_ID = "1M17"
CHAIN_ID = "A"      # None이면 모든 단백질 chain 유지
PH = 7.4

# ==============================
# 1. PDB 다운로드
# ==============================
url = f"https://files.rcsb.org/download/{PDB_ID.upper()}.pdb"
response = requests.get(url, timeout=60)
response.raise_for_status()

raw_pdb = f"{PDB_ID}_original.pdb"
with open(raw_pdb, "w") as f:
    f.write(response.text)

print("Downloaded:", raw_pdb)

# ==============================
# 2. 원본 Ligand 확인
# ==============================
hetero_residues = set()

for line in response.text.splitlines():
    if line.startswith("HETATM"):
        name = line[17:20].strip()
        chain = line[21].strip()
        number = line[22:27].strip()

        if name not in ["HOH", "WAT"]:
            hetero_residues.add((name, chain, number))

print("Original heterogens:", sorted(hetero_residues))

# ==============================
# 3. 구조 불러오기
# ==============================
fixer = PDBFixer(pdbfile=StringIO(response.text))

# 선택한 chain만 유지
if CHAIN_ID is not None:
    chain_indices = [
        i for i, chain in enumerate(fixer.topology.chains())
        if chain.id != CHAIN_ID
    ]

    fixer.removeChains(chain_indices)

# ==============================
# 4. 비표준 아미노산 처리
# ==============================
fixer.findNonstandardResidues()
print("Nonstandard residues:", fixer.nonstandardResidues)

fixer.replaceNonstandardResidues()

# ==============================
# 5. Ligand, water, 이온 제거
# ==============================
fixer.removeHeterogens(False)

# ==============================
# 6. Missing atoms 보완
# ==============================
fixer.findMissingResidues()

# 긴 missing loop는 자동 생성하지 않음
fixer.missingResidues = {}

fixer.findMissingAtoms()
fixer.addMissingAtoms()

# ==============================
# 7. Hydrogen 추가
# ==============================
fixer.addMissingHydrogens(PH)

# ==============================
# 8. PDB 저장
# ==============================
prepared_pdb = f"{PDB_ID}_prepared.pdb"

with open(prepared_pdb, "w") as f:
    PDBFile.writeFile(
        fixer.topology,
        fixer.positions,
        f,
        keepIds=True
    )

print("Prepared protein:", prepared_pdb)
print("Atoms:", fixer.topology.getNumAtoms())
print("Residues:", fixer.topology.getNumResidues())


Downloaded: 1M17_original.pdb
Original heterogens: [('AQ4', 'A', '999')]
Nonstandard residues: []
Prepared protein: 1M17_prepared.pdb
Atoms: 5029
Residues: 312


# Binding Site Prediction

In [2]:
!apt-get update -y
!apt-get install -y openjdk-11-jre-headless
!java -version
!wget https://github.com/rdk/p2rank/releases/download/2.5.1/p2rank_2.5.1.tar.gz
!tar -xzf p2rank_2.5.1.tar.gz

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:2 http://archive.ubuntu.com/ubuntu noble InRelease
Hit:3 http://security.ubuntu.com/ubuntu noble-security InRelease
Hit:4 http://archive.ubuntu.com/ubuntu noble-updates InRelease
Hit:5 https://r2u.stat.illinois.edu/ubuntu noble InRelease
Hit:6 http://archive.ubuntu.com/ubuntu noble-backports InRelease
Hit:7 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Fetched 3,917 B in 1s (5,146 B/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
openjdk-11-jre-headless is already the newest version (11.0.32.1+1-1ubuntu1~24.04).
0 upgraded, 0 newly installed, 0 to remove

In [4]:
!./p2rank_2.5.1/prank predict -f EXP.pdb -o output_folder_EXP # .pdb 파일 이름 수정 필요
# !./p2rank_2.5.1/prank predict -f PDB.pdb -o output_folder_PDB # .pdb 파일 이름 수정 필요

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
P2Rank 2.5.1
[INFO] Console - P2Rank 2.5.1

[INFO] Console - 
[INFO] Main - loading default config from [/content/p2rank_2.5.1/config/default.groovy]
DIR: /content/p2rank_2.5.1/config/../test_data
[INFO] Console - DIR: /content/p2rank_2.5.1/config/../test_data
DIR2: /content/p2rank_2.5.1/config/../test_data
[INFO] Console - DIR2: /content/p2rank_2.5.1/config/../test_data
DIR: /content/p2rank_2.5.1/config/../test_output
[INFO] Console - DIR: /content/p2rank_2.5.1/config/../test_output
DIR2: /content/p2rank_2.5.1/config/../test_output
[INFO] Console - DIR2: /content/p2rank_2.5.1/config/../test_output
predicting pockets for proteins from dataset [EXP.pdb]
[INFO] Con

In [ ]:
!tar -czf output_folder_EXP.tar.gz ./output_folder_EXP/
!tar -czf output_folder_PDB.tar.gz ./output_folder_PDB/